# Dynamic JSCC — train the **projection** site on Kaggle

Runs the model straight from
[`quantum-dynamic-jscc`](https://github.com/rowshan-mannan-oni/quantum-dynamic-jscc).
This notebook holds **no copy of the model**: it clones the repository and calls its
scripts, so everything reflects the latest commit.

This is **site B**, the channel-encoder projection — the `Conv3x3(256->16)` whose output
*is* the latent that gets power-normalised, masked and put on the channel. It is the site
with the strongest claim in the pipeline: every other one shapes or selects what a
classical layer transmits, this one produces the symbols.

| `ARM` | Module at the projection site | Flags added |
|---|---|---|
| `"quantum"` | `HybridVQC` — variational circuit, 2,232 params | `--quantum_site projection --n_qubits 8 --vqc_layers 2` |
| `"classical"` | the original `Conv3x3`, 36,880 params | none |
| `"matched"` | `MatchedBottleneck` — classical control, 2,272 params | `--matched_site projection --n_qubits 8` |

### Two properties of this site to keep in view

**Both swapped arms are 1x1 where the classical reference is 3x3.** A circuit consumes one
vector per shot, so the 8x8 grid is folded into the batch and each position is mapped
independently — which gives up the neighbourhood the convolution sees. It applies equally to
the quantum and matched arms, so that comparison stays clean, but part of any gap against
the unmodified classical reference is the lost receptive field rather than the circuit.

**The transmitted latent is rank-limited to 8.** Each position produces 16 channels from a
`Linear(8->16)` applied to 8 PauliZ expectations, so those 16 values span an 8-dimensional
subspace. That is not purely a loss — it is a repetition code, and the decoder can average
noise over the redundant dimensions. It predicts that this arm tracks classical at low SNR,
where the channel is noise-limited, and falls behind at high SNR, where it is
dimension-limited. Section 8 measures it directly.

### Before you run
1. **Settings -> Accelerator -> GPU** (T4 or P100).
2. **Settings -> Internet -> On.** Required — the notebook clones from GitHub, and the
   quantum arm installs `torchquantum` from source.
3. **Add Input ->** a CIFAR-10 dataset (optional; it downloads otherwise).
4. Check **`BRANCH`** in the config cell. The projection site is not on `main` yet, so it
   defaults to `site-b-projection`; change it to `main` once that has been merged.

> **This arm does not fit in one session at the full schedule.** The quantum projection arm
> measures 1.1 min/epoch on an RTX 4070 Laptop, so ~7.3 h for 400 epochs, and a T4 is slower
> still. Plan on resuming at least once — section 5 has the procedure. Leave `QUICK_TEST =
> True` for the first pass.

## 1. Configuration — the only cell you normally edit

In [ ]:
# ---------------- source ----------------
# The projection site lives on this branch until it is merged. Set it to "main"
# once it has been - a stale branch name here silently trains the wrong code, or
# fails at --quantum_site projection if the site does not exist on that branch.
BRANCH = "site-b-projection"

# ---------------- which arm ----------------
ARM = "quantum"          # "quantum" | "classical" | "matched"

QUICK_TEST = True        # True: 8 epochs, to prove the pipeline end to end and see the
                         # diagnostics fire. False: the full 400-epoch paper schedule.

# ------------- experiment identity -------------
# These decide the checkpoint folder name, so training and the figure step must agree.
# Anything being compared must share the same SEED and VAL_SIZE, or the arms are not
# trained on the same data.
SEED          = 0
VAL_SIZE      = 2000     # images held out to score each epoch and keep the best checkpoint
LAMBDA_REWARD = 1.5e-3   # rate penalty; sweep to trace the rate-distortion curve
LAMBDA_L2     = 1.0      # keep this a float: it goes into the folder name
SELECT        = "hard"
C_CHANNEL     = 16

# ---------------- circuit ----------------
N_QUBITS   = 8           # cost grows as 2**N_QUBITS; see the note in section 8 on why
                         # more qubits is not an option at this site
VQC_LAYERS = 2           # circuit depth. Deeper is NOT safer - barren plateaus.

# ---------------- training ----------------
BATCH_SIZE      = 128
SNR_MIN         = 0
SNR_MAX         = 20
NUM_WORKERS     = 2      # Kaggle is Linux, so workers are safe here
SAVE_EPOCH_FREQ = 5

if QUICK_TEST:
    N_JOINT, N_DECAY, N_FINE = 4, 3, 1
else:
    N_JOINT, N_DECAY, N_FINE = 150, 150, 100

# ---------------- figures ----------------
NUM_TEST = 10000
SNR_LIST = "0,2,4,6,8,10,12,14,16,18,20"
FIG6_SNR = 10

SITE = "projection"      # what this notebook trains; the policy site has its own notebook
print(f"arm={ARM} site={SITE} | {N_JOINT + N_DECAY + N_FINE} epochs "
      f"({'quick test' if QUICK_TEST else 'full schedule'})")

## 2. Clone the repository, on the right branch

`BRANCH` in the config cell decides which code runs. The projection site is not on `main`
yet, so this must check the branch out rather than take the clone default — otherwise
`--quantum_site projection` fails with *no swappable hook for site(s) ['projection']*, which
is the validation doing its job but three cells later than you want to find out.

The checkout is unconditional, so a session that already has the repository from an earlier
run switches branches rather than silently staying on whatever it cloned first.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rowshan-mannan-oni/quantum-dynamic-jscc.git"
REPO_DIR = "/kaggle/working/quantum-dynamic-jscc"

if os.path.isdir(REPO_DIR):
    print("Repository present - fetching")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--prune"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Unconditional, so re-running in a session that cloned a different branch still lands
# on BRANCH. Fails loudly if the branch does not exist rather than falling back to main.
subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("\nWorking directory:", os.getcwd())
subprocess.run(["git", "-C", REPO_DIR, "log", "-1",
                f"--format=Using {BRANCH} @ %h - %s"], check=True)

# The site has to exist in the code that was just checked out. Catching it here costs a
# second; catching it in train_dyna.py costs the dependency install and the data staging.
if ARM != "classical":
    from models.networks import SWAPPABLE_SITES
    assert SITE in SWAPPABLE_SITES, (
        f"branch {BRANCH!r} has no {SITE!r} hook (found: {list(SWAPPABLE_SITES)}). "
        f"Check BRANCH in the config cell.")
    print(f"site {SITE!r} is available on this branch")

## 3. Dependencies

Kaggle already ships a CUDA build of PyTorch. **Do not** install the pinned torch from
`requirements.txt` — that would replace a working GPU install.

The quantum arm additionally needs `torchquantum`, installed from GitHub with `--no-deps`.
The reasons are in `requirements-quantum.txt`: the last PyPI release imports a qiskit path
removed in 1.0, and the declared dependency list drags in tensorflow and pyscf, which
statevector simulation never touches.

In [ ]:
import importlib, subprocess, sys

def ensure(module, install_args, label=None):
    label = label or module
    if importlib.util.find_spec(module) is None:
        print(f"installing {label} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + install_args, check=True)
    else:
        print(f"{label} already available")

ensure("skimage", ["scikit-image"], "scikit-image")
ensure("matplotlib", ["matplotlib"])

if ARM == "quantum":
    ensure("torchquantum",
           ["--no-deps", "git+https://github.com/mit-han-lab/torchquantum.git"],
           "torchquantum (no-deps)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements-quantum.txt"], check=True)

import torch
print("\ntorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Enable it under Settings -> Accelerator.")

GPU_IDS = "0" if torch.cuda.is_available() else "-1"

### Verify the circuit *and this site's wiring* before committing hours to it

`quantum.smoke_test` now goes past the install check. It builds **every registered site**
through the same `build_swappable` hook training uses, for all three arms, at the real batch
and shapes, and asserts that the output shape is unchanged, that gradients reach the circuit
angles, and that `PerPosition` — the fold that turns the 8x8 grid into a batch of 8192 — is
exactly a 1x1 convolution.

That last check matters here specifically: getting its permute and reshape the wrong way
round transposes positions against channels *without changing any shape*, so nothing would
raise. The model would simply train on scrambled spatial structure and report a worse number,
which at this site would read as "the circuit underperforms".

Expect `projection` to report classical 36,880 / matched 2,272 / quantum 2,232 parameters.
Two minutes here against a run of several hours.

In [ ]:
import subprocess, sys

if ARM == "quantum":
    # smoke_test takes --gpu <int> with a separate --cpu, not the training
    # scripts' --gpu_ids convention where -1 means CPU.
    where = ["--gpu", "0"] if GPU_IDS != "-1" else ["--cpu"]
    subprocess.run([sys.executable, "-m", "quantum.smoke_test"] + where, check=True)
else:
    print(f'ARM is "{ARM}" - no circuit to check.')

## 4. CIFAR-10

`train_dyna.py` reads from `./data` inside the repository, so an attached Kaggle dataset is
staged there. Falls back to downloading if nothing is attached.

In [ ]:
import glob, os, shutil, tarfile

DATA_ROOT = os.path.join(REPO_DIR, "data")
os.makedirs(DATA_ROOT, exist_ok=True)
target = os.path.join(DATA_ROOT, "cifar-10-batches-py")

def find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

if not os.path.isdir(target):
    folder = find("/kaggle/input/**/cifar-10-batches-py")
    tar = find("/kaggle/input/**/cifar-10-python.tar.gz")
    if folder:
        print("Found extracted dataset:", folder)
        shutil.copytree(folder, target)
    elif tar:
        print("Found tarball:", tar)
        with tarfile.open(tar) as t:
            t.extractall(DATA_ROOT)
    else:
        print("No CIFAR-10 under /kaggle/input - it will download (Internet must be On).")

print("Dataset ready:", os.path.isdir(target))

## 5. The run folder, the figure folder, and resuming

The checkpoint folder name is **derived from the flags**, not chosen — `run_name` below is
imported from the repository so this notebook cannot disagree with what the training script
writes. Nothing takes a checkpoint path, so evaluation later has to repeat the same flags.

**The figure folder mirrors it.** Figures go to `figures/<same name>/` rather than to a label
like `figures/quantum/`, so a set of plots can always be traced back to the exact checkpoint
and flag combination that produced it — including the qubit count, depth and seed, none of
which a hand-written label carries.

The full schedule at this site will outlast a Kaggle session. To continue one: download this
notebook's output, upload the `Checkpoints/<run>/` folder as a Kaggle Dataset, attach it with
**Add Input**, and re-run. The cell finds it, copies it back, and training resumes from the
last rolling checkpoint.

In [ ]:
import glob, os, re, shutil
from types import SimpleNamespace

from options.base_options import run_name

ARM_FLAGS = {
    "classical": [],
    "matched":   ["--matched_site", SITE, "--n_qubits", str(N_QUBITS)],
    "quantum":   ["--quantum_site", SITE, "--n_qubits", str(N_QUBITS),
                  "--vqc_layers", str(VQC_LAYERS)],
}[ARM]

# Flags that identify the experiment. Training and every evaluation step must pass the
# same ones, because they are what rebuilds the folder name.
IDENTITY_FLAGS = ["--seed", str(SEED),
                  "--select", SELECT,
                  "--C_channel", str(C_CHANNEL),
                  "--lambda_L2", str(LAMBDA_L2),
                  "--lambda_reward", str(LAMBDA_REWARD)]

# Same fields run_name() reads, so the notebook and the scripts agree by construction.
CKPT_NAME = run_name(SimpleNamespace(
    C_channel=C_CHANNEL, lambda_L2=LAMBDA_L2, lambda_reward=LAMBDA_REWARD, select=SELECT,
    quantum_site=SITE if ARM == "quantum" else "none",
    matched_site=SITE if ARM == "matched" else "none",
    n_qubits=N_QUBITS, vqc_layers=VQC_LAYERS, seed=SEED))

CKPT_DIR = os.path.join("Checkpoints", CKPT_NAME)
FIG_DIR = os.path.join("/kaggle/working/figures", CKPT_NAME)   # mirrors the checkpoint name

print("run   :", CKPT_NAME)
print("ckpt  :", CKPT_DIR)
print("figs  :", FIG_DIR)

# Restore a previous session's checkpoint, if one is attached.
RESUME, START_EPOCH = False, 1
if not os.path.isdir(CKPT_DIR):
    prior = next(iter(glob.glob(f"/kaggle/input/**/{CKPT_NAME}", recursive=True)), None)
    if prior:
        print("\nFound a previous run under /kaggle/input:", prior)
        shutil.copytree(prior, CKPT_DIR)
    else:
        print("\nNo previous checkpoint found - starting from scratch.")

if os.path.exists(os.path.join(CKPT_DIR, "latest_net_CE.pth")):
    RESUME = True
    log = os.path.join(CKPT_DIR, "loss_log.txt")
    done = 0
    if os.path.exists(log):
        seen = [int(m.group(1)) for m in
                (re.search(r"epoch:\s*(\d+)", l) for l in open(log)) if m]
        done = max(seen) if seen else 0
    START_EPOCH = done + 1
    print(f"Resuming: {done} epochs already logged, continuing from epoch {START_EPOCH}")

## 6. Train

Output is streamed live. `train_dyna.py` writes `latest_*` every `SAVE_EPOCH_FREQ` epochs and
`best_*` whenever the held-out score improves, plus `opt.txt` recording the exact command.

Unlike the policy site, the projection lives in `netCE`, which the fine-tuning stage does
**not** freeze — so a circuit here keeps training for the whole schedule instead of stopping
at epoch 300.

In [ ]:
import subprocess, sys, time

cmd = [sys.executable, "-u", "train_dyna.py",
       "--gpu_ids", GPU_IDS,
       "--val_size", str(VAL_SIZE),
       "--batch_size", str(BATCH_SIZE),
       "--SNR_MIN", str(SNR_MIN), "--SNR_MAX", str(SNR_MAX),
       "--n_epochs_joint", str(N_JOINT),
       "--n_epochs_decay", str(N_DECAY),
       "--n_epochs_fine", str(N_FINE),
       "--num_workers", str(NUM_WORKERS),
       "--save_epoch_freq", str(SAVE_EPOCH_FREQ),
       "--print_freq", "12800"] + IDENTITY_FLAGS + ARM_FLAGS

if RESUME:
    cmd += ["--continue_train", "--epoch_count", str(START_EPOCH)]

print(" ".join(cmd), "\n")
start = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"training failed with exit code {proc.returncode}")
print(f"\nTraining finished in {(time.time() - start) / 3600:.2f} h")

## 7. Did it train, and is the rate still adaptive?

Two different questions, and at this site they mean different things than they did at the
policy site.

`G_L2` is the reconstruction term — it should fall and flatten. If it plateaus early and
high, the 1x1 rank-8 bottleneck is binding hard.

`G_reward` is the mean number of active selective groups. **The policy network here is the
unmodified classical MLP**, so unlike the policy-site notebook a constant `G_reward` does not
mean the policy collapsed — it means the policy is being handed a latent it cannot
distinguish between images, which points at the projection, not at the gate.

In [ ]:
import os, re
import matplotlib.pyplot as plt
import numpy as np

log_path = os.path.join(CKPT_DIR, "loss_log.txt")
pat = re.compile(r"epoch:\s*(\d+).*?G_L2:\s*([\d.eE+-]+)\s+G_reward:\s*([\d.eE+-]+)")
rows = [(int(m.group(1)), float(m.group(2)), float(m.group(3)))
        for m in (pat.search(l) for l in open(log_path)) if m]
arr = np.array(rows)

last = arr[arr[:, 0] == arr[:, 0].max()]
spread = last[:, 2].std()
print(f"final epoch: {int(last[0, 0])} | G_L2 {last[:, 1].mean():.5f} "
      f"| G_reward {last[:, 2].mean():.5f} | std across batches {spread:.3e}")
if spread < 1e-5:
    print("\n*** RATE IS CONSTANT ***\n"
          "Every batch chose the same number of groups. The policy here is the untouched\n"
          "classical MLP, so this points at the projection handing it a latent that does\n"
          "not vary with the image. Check section 8 before spending a full run on this.")
else:
    print("\nRate still varies across batches - healthy so far.")

epochs = np.unique(arr[:, 0])
l2 = np.array([arr[arr[:, 0] == e, 1].mean() for e in epochs])
rw = np.array([arr[arr[:, 0] == e, 2].mean() for e in epochs])
psnr = 10 * np.log10(4.0 / np.clip(l2, 1e-12, None))      # images in [-1,1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, psnr, color="#2a78d6")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("train PSNR (dB)")
axes[0].set_title("Reconstruction")
axes[1].plot(epochs, rw, color="#eb6834")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("active selective groups")
axes[1].set_title("Rate chosen")
for ax in axes:
    ax.grid(alpha=.3)
    for boundary in (N_JOINT, N_JOINT + N_DECAY):     # learning-rate stage changes
        if boundary < epochs.max():
            ax.axvline(boundary, color="#6b6a63", lw=.8, ls="--")
fig.suptitle(CKPT_NAME, fontsize=9)
fig.tight_layout(); plt.show()

## 8. How much is the rank-8 ceiling actually costing?

This is the diagnostic specific to this site.

Each position emits 16 latent channels built by `Linear(8->16)` from 8 PauliZ expectations,
so the transmitted values at a position can span **at most 8 dimensions** of the 16
available. The cell below measures what is actually used: it runs a batch through
`netSE -> netCE`, stacks every spatial position as a 16-vector, and takes the singular
values.

**Reading it.** Sixteen roughly comparable singular values would mean the channels are being
used independently — that is what the classical arm should look like. A sharp cliff after the
8th confirms the ceiling is binding exactly as predicted. A cliff *well before* the 8th is
the bad case: it means the circuit is not even using the eight dimensions it has, which
points at angle-encoder saturation rather than at the rank limit, and the saturation figure
below tells you which.

Note that raising `N_QUBITS` is not the escape route it looks like. A statevector at batch
8192 costs `2**q * 8192 * 8` bytes — 17 MB at 8 qubits, 268 MB at 12, **4.3 GB at 16** — and
autograd retains roughly 48 of them. Folding 64 positions into the batch is what makes this
site affordable at all, and it is exactly what forecloses widening the state. Reading more
observables out of the same 8-qubit state (PauliX and PauliY alongside Z) is the available
lever, and it is a change to the readout rather than to the circuit.

In [ ]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

from make_figures import FigureOptions      # has --dataroot and --eval_batch
from models import create_model

argv_backup = sys.argv
sys.argv = (["diagnostic", "--gpu_ids", GPU_IDS, "--dataroot", DATA_ROOT,
             "--eval_batch", "250"] + IDENTITY_FLAGS + ARM_FLAGS)
opt = FigureOptions().parse()
sys.argv = argv_backup

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
testset = torchvision.datasets.CIFAR10(root=opt.dataroot, train=False,
                                       download=True, transform=transform)
images, _ = next(iter(torch.utils.data.DataLoader(testset, batch_size=250, shuffle=False)))

model = create_model(opt)
model.setup(opt)
model.eval()

images = images.to(model.device)
snr = torch.full((images.shape[0], 1), float(FIG6_SNR), device=model.device)

# Capture the pre-linear output so the tanh saturation can be measured on the way past.
netCE = model.netCE.module if hasattr(model.netCE, "module") else model.netCE
site_module = netCE.projection
inner = getattr(site_module, "module", None)        # PerPosition -> HybridVQC / matched
captured = {}
handle = (inner.pre.register_forward_hook(lambda m, i, o: captured.__setitem__("pre", o.detach()))
          if inner is not None else None)

with torch.no_grad():
    latent = netCE(model.netSE(images), snr)        # (N, 16, 8, 8)
if handle is not None:
    handle.remove()

# Every spatial position of every image, as a 16-vector.
positions = latent.permute(0, 2, 3, 1).reshape(-1, latent.shape[1]).float()
positions = positions - positions.mean(0, keepdim=True)
sv = torch.linalg.svdvals(positions).cpu().numpy()
energy = np.cumsum(sv ** 2) / np.sum(sv ** 2)
eff_rank = int(np.searchsorted(energy, 0.99) + 1)

print(f"latent {tuple(latent.shape)} -> {positions.shape[0]:,} positions of "
      f"{positions.shape[1]} channels")
print(f"dimensions holding 99% of the energy: {eff_rank} of {latent.shape[1]}")
if inner is not None:
    print(f"expected ceiling for this arm: {N_QUBITS}")
if "pre" in captured:
    sat = (torch.tanh(captured["pre"]).abs() > 0.99).float().mean().item()
    print(f"angle-encoder saturation (|tanh| > 0.99): {sat:.1%}")
    if sat > 0.5:
        print("  *** more than half the encoder is pinned - the circuit is barely "
              "responding to its input ***")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(np.arange(1, len(sv) + 1), sv / sv[0], color="#2a78d6")
axes[0].set_xlabel("component"); axes[0].set_ylabel("singular value (relative)")
axes[0].set_title("Latent spectrum per position")
axes[1].plot(np.arange(1, len(sv) + 1), energy, marker="o", color="#eb6834")
axes[1].axhline(0.99, color="#6b6a63", lw=.8, ls="--")
axes[1].set_xlabel("components kept"); axes[1].set_ylabel("cumulative energy")
axes[1].set_title("How many dimensions are actually used")
for ax in axes:
    ax.grid(alpha=.3)
    if inner is not None:
        ax.axvline(N_QUBITS + .5, color="#1baf7a", lw=1,
                   label=f"{N_QUBITS}-qubit ceiling")
        ax.legend(fontsize=8)
fig.suptitle(CKPT_NAME, fontsize=9)
fig.tight_layout(); plt.show()

## 9. Figures

Same flags as training — the folder name is rebuilt from them, so they have to match.
Written to `figures/<checkpoint name>/`, so the plots and the weights that produced them
carry the same identifier.

In [ ]:
import subprocess, sys

cmd = [sys.executable, "-u", "make_figures.py",
       "--gpu_ids", GPU_IDS,
       "--num_test", str(NUM_TEST),
       "--snr_list", SNR_LIST,
       "--fig6_snr", str(FIG6_SNR),
       "--num_workers", str(NUM_WORKERS),
       "--dataroot", DATA_ROOT,
       "--results_dir", FIG_DIR] + IDENTITY_FLAGS + ARM_FLAGS

print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"make_figures.py failed with exit code {proc.returncode}")

In [ ]:
from IPython.display import Image, display, Markdown
import os

titles = {
    "fig4_rate_psnr_vs_snr.png": "Average rate (CPP) and PSNR vs SNR",
    "fig5_attainable_psnr.png":  "Attainable PSNR: fixed rates vs adaptive",
    "fig6_per_class.png":        "Per-class rate and PSNR",
    "reconstructions.png":       "Sample reconstructions",
}
display(Markdown(f"**{CKPT_NAME}**"))
for name, title in titles.items():
    path = os.path.join(FIG_DIR, name)
    if os.path.exists(path):
        display(Markdown(f"### {title}"))
        display(Image(filename=path))
    else:
        print("missing:", path)

## 10. Compare against any other arm you have

Picks up every run that has a `fig4_data.csv`, whether it was produced in this session under
`/kaggle/working/figures/` or attached as an input from an earlier one. Because the figure
folders are named after their checkpoints, each curve is labelled with the run that produced
it rather than with a generic arm name — so a classical reference, the policy site and this
one can all sit on the same axes without ambiguity.

The two panels are kept separate on purpose: two scales on one frame make the crossing point
of the curves an artefact of the scaling, and where the arms separate is the whole question.

In [ ]:
import glob, os, re, subprocess, sys

found = {}
for pattern in ("/kaggle/working/figures/*/fig4_data.csv",
                "/kaggle/input/**/fig4_data.csv"):
    for path in glob.glob(pattern, recursive=True):
        found.setdefault(os.path.basename(os.path.dirname(path)), os.path.dirname(path))

def label(name):
    """Turn a run folder name into a legend entry.

    Reads back exactly what run_name() encodes - arm, site, qubits, depth, seed -
    so a run that differs only in depth or seed still gets a distinct label
    instead of two curves both called "quantum".
    """
    arm = re.search(r"_(q|m)([a-z_+]+)-q(\d+)(?:l(\d+))?", name)
    seed = re.search(r"_s(\d+)$", name)
    tag = f" s{seed.group(1)}" if seed else ""
    if not arm:
        return "classical" + tag
    kind = "quantum" if arm.group(1) == "q" else "matched"
    depth = f"l{arm.group(4)}" if arm.group(4) else ""
    return f"{kind} {arm.group(2)} q{arm.group(3)}{depth}{tag}"

if len(found) < 2:
    print("Only", len(found), "run(s) with figures:", list(found))
    print("Train another arm, or attach an earlier run's figures as an input, then re-run.")
else:
    runs = ",".join(f"{label(n)}={d}" for n, d in sorted(found.items()))
    print("comparing:", runs, "\n")
    subprocess.run([sys.executable, "compare_arms.py",
                    "--runs", runs,
                    "--title", f"Site B - {SITE}",
                    "--results_dir", "/kaggle/working/figures"], check=True)
    from IPython.display import Image, display
    display(Image(filename="/kaggle/working/figures/comparison_rate_psnr.png"))

## Notes

- **Everything is under `/kaggle/working`**, so checkpoints, `opt.txt`, logs and figures all
  appear in the notebook's Output tab. `opt.txt` records the exact command that trained the
  run — the flags you will need to evaluate it later.
- **Figure folders mirror checkpoint folders.** `figures/C16_L2_1.0_re_0.0015_hard_qprojection-q8l2_s0/`
  came from `Checkpoints/C16_L2_1.0_re_0.0015_hard_qprojection-q8l2_s0/`, and nothing else.
  The qubit count, depth and seed are all in the name, which a label like `figures/quantum/`
  cannot carry.
- **Runtimes.** The quantum arm at this site measures 1.1 min/epoch on an RTX 4070 Laptop
  against 0.6 for the matched arm and 0.5 classical — roughly 7.3 h for 400 epochs, and
  longer on a T4. Expect to resume at least once; section 5 does it automatically once the
  previous `Checkpoints/<run>/` is attached as an input.
- **The classical reference is shared across sites.** You do not retrain it per site — train
  it once at the same `SEED` and `VAL_SIZE` and reuse it.
- Full argument reference: `RUNS.md` in the repository. Why the experiments are shaped this
  way: `Quantum_JSCC_Plan.md`.